# DS107 IAA After Guideline Update Analysis

Notebook này phân tích vòng pilot sau khi cập nhật guideline từ file `data/04_Labeling_Pilot/Topic Annotation Pilot Set 100 [2] - IAA_Merge.csv`.

Nội dung tập trung vào Fleiss' kappa, Cohen's kappa từng cặp, phân phối nhãn, nhãn không dùng, tần suất đồng thuận/bất đồng, nhãn hay phát sinh bất đồng, cặp nhãn hay bất đồng và mức giống/khác nhau giữa annotator. Notebook không ghi đè notebook/report IAA cũ.


In [ ]:
from pathlib import Path
from collections import Counter
from itertools import combinations
from datetime import datetime
import math
import pandas as pd

# ---------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------
CWD = Path.cwd()
if (CWD / "data" / "04_Labeling_Pilot").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "data" / "04_Labeling_Pilot").exists():
    REPO_ROOT = CWD.parent
else:
    raise FileNotFoundError("Không tìm thấy data/04_Labeling_Pilot từ current directory hoặc parent directory.")

DATA_DIR = REPO_ROOT / "data" / "04_Labeling_Pilot"
AFTER_GUIDELINE_PATH = DATA_DIR / "Topic Annotation Pilot Set 100 [2] - IAA_Merge.csv"
REPORT_PATH = REPO_ROOT / "iaa_analysis_after_update_guideline.md"

ANNOTATOR_COLS = {
    "Nhung": "annotator_Nhung_topic_label",
    "Yen": "annotator_Yen_topic_label",
    "Han": "annotator_Han_topic_label",
}
ANNOTATORS = list(ANNOTATOR_COLS.keys())
AGREEMENT_ORDER = ["full_agreement", "partial_disagreement", "complete_disagreement", "incomplete"]

DEFAULT_LABEL_TO_ID = {
    "POLITICS": "T01",
    "ECONOMY_BUSINESS_AND_FINANCE": "T02",
    "CRIME_LAW_AND_JUSTICE": "T03",
    "HEALTH": "T04",
    "EDUCATION": "T05",
    "SCIENCE_AND_TECHNOLOGY": "T06",
    "ENVIRONMENT": "T07",
    "WEATHER": "T08",
    "DISASTER_ACCIDENT_AND_EMERGENCY_INCIDENT": "T09",
    "ARTS_CULTURE_ENTERTAINMENT_AND_MEDIA": "T10",
    "SPORT": "T11",
    "SOCIETY": "T12",
    "HUMAN_INTEREST": "T13",
    "LABOR": "T14",
    "LIFESTYLE_AND_LEISURE": "T15",
    "WORLD_INTERNATIONAL": "T16",
    "TRANSPORT_INFRASTRUCTURE": "T17",
    "OTHER_UNCLEAR": "T18",
}


def normalize_label(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value if value else None


def fmt_pct(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "NA"
    return f"{value * 100:.1f}%"


def fmt_num(value, digits=4):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "NA"
    return f"{value:.{digits}f}"


def clean_cell(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return str(value).replace("\n", " ").replace("|", "\\|")


def md_table(rows, headers):
    if isinstance(rows, pd.DataFrame):
        headers = list(rows.columns)
        rows = rows.to_dict("records")
    if not rows:
        return "_Không có._"
    if isinstance(rows[0], dict):
        body = [[clean_cell(row.get(h, "")) for h in headers] for row in rows]
    else:
        body = [[clean_cell(v) for v in row] for row in rows]
    header = "| " + " | ".join(clean_cell(h) for h in headers) + " |"
    sep = "| " + " | ".join("---" for _ in headers) + " |"
    lines = [header, sep]
    lines += ["| " + " | ".join(row) + " |" for row in body]
    return "\n".join(lines)


def read_pilot(path):
    df = pd.read_csv(path)
    missing_cols = [col for col in ANNOTATOR_COLS.values() if col not in df.columns]
    if missing_cols:
        raise ValueError(f"File {path.name} thiếu cột annotator: {missing_cols}")
    df = df.copy()
    df["source_file"] = path.name
    df["row_in_file"] = range(1, len(df) + 1)
    return df


def extract_label_frame(df):
    labels = df[list(ANNOTATOR_COLS.values())].copy()
    labels.columns = ANNOTATORS
    for col in labels.columns:
        labels[col] = labels[col].map(normalize_label)
    return labels


def build_label_universe(df):
    labels = set(DEFAULT_LABEL_TO_ID.keys())
    label_to_id = dict(DEFAULT_LABEL_TO_ID)
    label_cols = list(ANNOTATOR_COLS.values()) + ["topic_label_final", "topic_label_final_auto", "topic_label_final_train"]
    for col in label_cols:
        if col in df.columns:
            labels.update(v for v in df[col].map(normalize_label).dropna().tolist())
    if "topic_label_final_train" in df.columns and "topic_label_final_id_train" in df.columns:
        for label, label_id in df[["topic_label_final_train", "topic_label_final_id_train"]].dropna().itertuples(index=False):
            label = normalize_label(label)
            label_id = normalize_label(label_id)
            if label and label_id:
                label_to_id[label] = label_id
    ordered = sorted(labels, key=lambda label: (label_to_id.get(label, "T99"), label))
    return ordered, label_to_id


def cohen_kappa(labels_a, labels_b, label_universe):
    pairs = [(a, b) for a, b in zip(labels_a, labels_b) if a is not None and b is not None]
    n = len(pairs)
    if n == 0:
        return {"n": 0, "observed_agreement": math.nan, "expected_agreement": math.nan, "kappa": math.nan}
    observed = sum(1 for a, b in pairs if a == b) / n
    counts_a = Counter(a for a, _ in pairs)
    counts_b = Counter(b for _, b in pairs)
    expected = sum((counts_a[label] / n) * (counts_b[label] / n) for label in label_universe)
    denom = 1 - expected
    kappa = 1.0 if denom == 0 and observed == 1 else ((observed - expected) / denom if denom != 0 else math.nan)
    return {"n": n, "observed_agreement": observed, "expected_agreement": expected, "kappa": kappa}


def fleiss_kappa(label_df, label_universe):
    valid = label_df.dropna()
    n_items = len(valid)
    n_raters = len(label_df.columns)
    if n_items == 0 or n_raters < 2:
        return {"n_items": n_items, "n_raters": n_raters, "observed_agreement": math.nan, "expected_agreement": math.nan, "kappa": math.nan}
    row_agreements = []
    label_totals = Counter()
    for row in valid.itertuples(index=False):
        counts = Counter(row)
        label_totals.update(counts)
        row_agreements.append(sum(count * (count - 1) for count in counts.values()) / (n_raters * (n_raters - 1)))
    observed = sum(row_agreements) / n_items
    total_assignments = n_items * n_raters
    expected = sum((label_totals[label] / total_assignments) ** 2 for label in label_universe)
    denom = 1 - expected
    kappa = 1.0 if denom == 0 and observed == 1 else ((observed - expected) / denom if denom != 0 else math.nan)
    return {"n_items": n_items, "n_raters": n_raters, "observed_agreement": observed, "expected_agreement": expected, "kappa": kappa}


def agreement_type(row):
    values = [v for v in row if v is not None]
    if len(values) < len(ANNOTATORS):
        return "incomplete"
    unique_count = len(set(values))
    if unique_count == 1:
        return "full_agreement"
    if unique_count == 2:
        return "partial_disagreement"
    return "complete_disagreement"


def majority_label(row):
    values = [v for v in row if v is not None]
    if len(values) < len(ANNOTATORS):
        return None
    counts = Counter(values)
    top_count = max(counts.values())
    if top_count >= 2:
        return sorted([label for label, count in counts.items() if count == top_count])[0]
    return "NO_MAJORITY_TIE"


def label_distribution(label_df, label_universe):
    rows = []
    for label in label_universe:
        row = {"label": label}
        row["total_assignments"] = int((label_df == label).sum().sum())
        for annotator in ANNOTATORS:
            row[annotator] = int((label_df[annotator] == label).sum())
        rows.append(row)
    out = pd.DataFrame(rows)
    total = out["total_assignments"].sum()
    out["assignment_pct"] = out["total_assignments"].map(lambda n: n / total if total else 0)
    return out.sort_values(["total_assignments", "label"], ascending=[False, True]).reset_index(drop=True)


def annotator_alignment_table(label_df, label_universe):
    valid = label_df.dropna().copy()
    rows = []
    totals = {annotator: int(label_df[annotator].notna().sum()) for annotator in ANNOTATORS}
    distributions = {}
    for annotator in ANNOTATORS:
        denom = totals[annotator]
        counts = label_df[annotator].value_counts()
        distributions[annotator] = {label: counts.get(label, 0) / denom if denom else 0 for label in label_universe}
    group_mean = {label: sum(distributions[a][label] for a in ANNOTATORS) / len(ANNOTATORS) for label in label_universe}

    for annotator in ANNOTATORS:
        others = [other for other in ANNOTATORS if other != annotator]
        pair_kappas = []
        pair_agreements = []
        for other in others:
            metric = cohen_kappa(label_df[annotator].tolist(), label_df[other].tolist(), label_universe)
            pair_kappas.append(metric["kappa"])
            pair_agreements.append(metric["observed_agreement"])
        if len(valid):
            other_consensus_mask = valid[others[0]] == valid[others[1]]
            n_other_consensus = int(other_consensus_mask.sum())
            n_match_other_consensus = int((valid.loc[other_consensus_mask, annotator] == valid.loc[other_consensus_mask, others[0]]).sum())
        else:
            n_other_consensus = 0
            n_match_other_consensus = 0
        n_minority = n_other_consensus - n_match_other_consensus
        l1_to_group = sum(abs(distributions[annotator][label] - group_mean[label]) for label in label_universe)
        rows.append({
            "annotator": annotator,
            "avg_pairwise_cohen": sum(pair_kappas) / len(pair_kappas) if pair_kappas else math.nan,
            "avg_pairwise_agreement": sum(pair_agreements) / len(pair_agreements) if pair_agreements else math.nan,
            "avg_pairwise_disagreement_rate": 1 - (sum(pair_agreements) / len(pair_agreements)) if pair_agreements else math.nan,
            "other2_consensus_rows": n_other_consensus,
            "match_other2_consensus_rows": n_match_other_consensus,
            "match_other2_consensus_rate": n_match_other_consensus / n_other_consensus if n_other_consensus else math.nan,
            "minority_when_other2_agree_rows": n_minority,
            "minority_when_other2_agree_rate": n_minority / n_other_consensus if n_other_consensus else math.nan,
            "label_distribution_l1_to_group_mean": l1_to_group,
        })
    out = pd.DataFrame(rows)
    similarity_order = out.sort_values(
        ["avg_pairwise_cohen", "match_other2_consensus_rate", "label_distribution_l1_to_group_mean"],
        ascending=[False, False, True],
    )["annotator"].tolist()
    outlier_order = out.sort_values(
        ["avg_pairwise_cohen", "match_other2_consensus_rate", "label_distribution_l1_to_group_mean"],
        ascending=[True, True, False],
    )["annotator"].tolist()
    out["average_similarity_rank"] = out["annotator"].map({name: idx + 1 for idx, name in enumerate(similarity_order)})
    out["outlier_rank"] = out["annotator"].map({name: idx + 1 for idx, name in enumerate(outlier_order)})
    return out.sort_values("average_similarity_rank").reset_index(drop=True)


def analyze_dataset(name, df, label_universe, label_to_id, row_scope):
    label_df = extract_label_frame(df)
    valid_mask = label_df.notna().all(axis=1)
    valid = label_df[valid_mask].copy()
    types = label_df.apply(lambda row: agreement_type(row.tolist()), axis=1)
    valid_types = types[valid_mask]
    valid_disagreement_mask = valid_types.isin(["partial_disagreement", "complete_disagreement"])
    disagreement_rows = valid[valid_disagreement_mask].copy()
    n_disagreement_rows = len(disagreement_rows)

    fleiss = fleiss_kappa(label_df, label_universe)
    cohen_rows = []
    for a, b in combinations(ANNOTATORS, 2):
        metric = cohen_kappa(label_df[a].tolist(), label_df[b].tolist(), label_universe)
        cohen_rows.append({
            "pair": f"{a} vs {b}",
            "n": metric["n"],
            "observed_agreement": metric["observed_agreement"],
            "expected_agreement": metric["expected_agreement"],
            "cohen_kappa": metric["kappa"],
        })
    cohen_df = pd.DataFrame(cohen_rows)

    type_counts = types.value_counts().reindex(AGREEMENT_ORDER, fill_value=0).reset_index()
    type_counts.columns = ["agreement_type", "rows"]
    type_counts["pct_all_rows"] = type_counts["rows"].map(lambda n: n / len(df) if len(df) else 0)

    used_labels = set(v for col in label_df.columns for v in label_df[col].dropna().tolist() if v is not None)
    unused_labels = [label for label in label_universe if label not in used_labels]
    dist = label_distribution(label_df, label_universe)

    majority = valid.apply(lambda row: majority_label(row.tolist()), axis=1)
    majority_counts = majority.value_counts().rename_axis("label").reset_index(name="rows")
    majority_counts["pct_valid_rows"] = majority_counts["rows"].map(lambda n: n / len(valid) if len(valid) else 0)

    label_rows = []
    for label in label_universe:
        rows_with_label = valid.apply(lambda row: label in set(row.tolist()), axis=1) if len(valid) else pd.Series(dtype=bool)
        rows_with_label_count = int(rows_with_label.sum()) if len(valid) else 0
        disagreement_with_label = int((rows_with_label & valid_disagreement_mask).sum()) if len(valid) else 0
        unanimous_for_label = int((valid == label).all(axis=1).sum()) if len(valid) else 0
        assignment_count = int((valid == label).sum().sum()) if len(valid) else 0
        disagreement_assignment_count = int((disagreement_rows == label).sum().sum()) if n_disagreement_rows else 0
        label_rows.append({
            "label": label,
            "label_id": label_to_id.get(label, ""),
            "assignment_count_valid": assignment_count,
            "rows_with_label": rows_with_label_count,
            "unanimous_rows": unanimous_for_label,
            "disagreement_rows_with_label": disagreement_with_label,
            "disagreement_assignment_count": disagreement_assignment_count,
            "disagreement_rate_when_label_appears": disagreement_with_label / rows_with_label_count if rows_with_label_count else math.nan,
        })
    label_stats = pd.DataFrame(label_rows)
    no_disagreement_labels = label_stats[(label_stats["rows_with_label"] > 0) & (label_stats["disagreement_rows_with_label"] == 0)].copy()
    frequent_disagreement_labels = label_stats[label_stats["disagreement_rows_with_label"] > 0].sort_values(
        ["disagreement_rows_with_label", "disagreement_rate_when_label_appears", "assignment_count_valid", "label"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)

    confusion_counter = Counter()
    for row in disagreement_rows.itertuples(index=False):
        for a, b in combinations(sorted(set(row)), 2):
            confusion_counter[(a, b)] += 1
    confusion_df = pd.DataFrame([
        {"label_a": a, "label_b": b, "disagreement_rows": count, "pct_of_disagreement_rows": count / n_disagreement_rows if n_disagreement_rows else 0}
        for (a, b), count in confusion_counter.most_common()
    ])

    return {
        "name": name,
        "row_scope": row_scope,
        "rows_total": len(df),
        "rows_valid_3_annotators": int(valid_mask.sum()),
        "rows_incomplete": int((~valid_mask).sum()),
        "labels_used_count": len(used_labels),
        "labels_unused_count": len(unused_labels),
        "unused_labels": unused_labels,
        "fleiss": fleiss,
        "cohen": cohen_df,
        "agreement_counts": type_counts,
        "assignment_distribution": dist,
        "majority_distribution": majority_counts,
        "annotator_alignment": annotator_alignment_table(label_df, label_universe),
        "label_stats": label_stats,
        "no_disagreement_labels": no_disagreement_labels.sort_values(["label_id", "label"]).reset_index(drop=True),
        "frequent_disagreement_labels": frequent_disagreement_labels,
        "confusion_pairs": confusion_df,
    }


def format_metric_tables(result):
    cohen = result["cohen"].copy()
    for col in ["observed_agreement", "expected_agreement", "cohen_kappa"]:
        cohen[col] = cohen[col].map(fmt_num)
    fleiss = result["fleiss"]
    fleiss_rows = [{
        "n_items": fleiss["n_items"],
        "n_raters": fleiss["n_raters"],
        "observed_agreement": fmt_num(fleiss["observed_agreement"]),
        "expected_agreement": fmt_num(fleiss["expected_agreement"]),
        "fleiss_kappa": fmt_num(fleiss["kappa"]),
    }]
    return fleiss_rows, cohen


def format_annotator_alignment(result):
    alignment = result["annotator_alignment"].copy()
    for col in ["avg_pairwise_cohen", "label_distribution_l1_to_group_mean"]:
        alignment[col] = alignment[col].map(fmt_num)
    for col in ["avg_pairwise_agreement", "avg_pairwise_disagreement_rate", "match_other2_consensus_rate", "minority_when_other2_agree_rate"]:
        alignment[col] = alignment[col].map(fmt_pct)
    return alignment[[
        "average_similarity_rank", "outlier_rank", "annotator", "avg_pairwise_cohen",
        "avg_pairwise_agreement", "avg_pairwise_disagreement_rate", "other2_consensus_rows",
        "match_other2_consensus_rows", "match_other2_consensus_rate",
        "minority_when_other2_agree_rows", "minority_when_other2_agree_rate",
        "label_distribution_l1_to_group_mean",
    ]]


def overview_table(result):
    counts = dict(zip(result["agreement_counts"]["agreement_type"], result["agreement_counts"]["rows"]))
    return pd.DataFrame([{
        "dataset": result["name"],
        "rows": result["rows_total"],
        "valid_rows": result["rows_valid_3_annotators"],
        "incomplete": result["rows_incomplete"],
        "fleiss_kappa": fmt_num(result["fleiss"]["kappa"]),
        "mean_cohen_kappa": fmt_num(result["cohen"]["cohen_kappa"].mean()),
        "used_labels": result["labels_used_count"],
        "unused_labels": result["labels_unused_count"],
        "full": counts.get("full_agreement", 0),
        "partial": counts.get("partial_disagreement", 0),
        "complete": counts.get("complete_disagreement", 0),
    }])


def result_to_markdown(result, top_n=20):
    fleiss_rows, cohen = format_metric_tables(result)
    alignment = format_annotator_alignment(result)

    agreement = result["agreement_counts"].copy()
    agreement["pct_all_rows"] = agreement["pct_all_rows"].map(fmt_pct)

    assignment_dist = result["assignment_distribution"].copy()
    assignment_dist["assignment_pct"] = assignment_dist["assignment_pct"].map(fmt_pct)

    majority_dist = result["majority_distribution"].copy()
    majority_dist["pct_valid_rows"] = majority_dist["pct_valid_rows"].map(fmt_pct)
    if result["rows_incomplete"]:
        majority_dist = pd.concat([
            majority_dist,
            pd.DataFrame([{"label": "INCOMPLETE_MISSING_LABELS", "rows": result["rows_incomplete"], "pct_valid_rows": "NA"}]),
        ], ignore_index=True)

    clean = result["no_disagreement_labels"][["label_id", "label", "rows_with_label", "assignment_count_valid"]]

    frequent = result["frequent_disagreement_labels"][[
        "label_id", "label", "rows_with_label", "disagreement_rows_with_label", "disagreement_rate_when_label_appears", "disagreement_assignment_count"
    ]].head(top_n).copy()
    frequent["disagreement_rate_when_label_appears"] = frequent["disagreement_rate_when_label_appears"].map(fmt_pct)

    pairs = result["confusion_pairs"].head(top_n).copy()
    if not pairs.empty:
        pairs["pct_of_disagreement_rows"] = pairs["pct_of_disagreement_rows"].map(fmt_pct)

    return "\n\n".join([
        f"## {result['name']}",
        f"Phạm vi dòng: {result['row_scope']}. Tổng dòng: {result['rows_total']}; dòng đủ 3 nhãn: {result['rows_valid_3_annotators']}; dòng thiếu nhãn: {result['rows_incomplete']}.",
        f"Nhãn được dùng: {result['labels_used_count']}/{len(LABEL_UNIVERSE)}. Nhãn không được dùng: {result['labels_unused_count']}" + (f" ({', '.join(result['unused_labels'])})." if result['unused_labels'] else "."),
        "### Fleiss' kappa",
        md_table(fleiss_rows, list(fleiss_rows[0].keys())),
        "### Cohen's kappa từng cặp",
        md_table(cohen, list(cohen.columns)),
        "### Annotator giống trung bình nhóm / khác biệt so với nhóm",
        md_table(alignment, list(alignment.columns)),
        "### Tần suất đồng thuận/bất đồng",
        md_table(agreement, list(agreement.columns)),
        "### Phân phối nhãn cấp dòng theo majority vote",
        md_table(majority_dist, list(majority_dist.columns)),
        "### Phân phối nhãn theo toàn bộ lượt gán",
        md_table(assignment_dist, list(assignment_dist.columns)),
        "### Nhãn không hề phát sinh bất đồng khi xuất hiện",
        md_table(clean, list(clean.columns)),
        "### Nhãn hay phát sinh bất đồng",
        md_table(frequent, list(frequent.columns)),
        "### Cặp nhãn hay bị bất đồng với nhau",
        md_table(pairs, list(pairs.columns) if not pairs.empty else ["label_a", "label_b", "disagreement_rows", "pct_of_disagreement_rows"]),
    ])


def build_report(result):
    overview = overview_table(result)
    label_rows = [{"label_id": LABEL_TO_ID.get(label, ""), "label": label} for label in LABEL_UNIVERSE]
    fleiss = result["fleiss"]
    cohen_mean = result["cohen"]["cohen_kappa"].mean()
    counts = dict(zip(result["agreement_counts"]["agreement_type"], result["agreement_counts"]["rows"]))
    disagreement_rate = (counts.get("partial_disagreement", 0) + counts.get("complete_disagreement", 0)) / result["rows_total"] if result["rows_total"] else math.nan
    top_majority = ", ".join(f"{row.label} ({int(row.rows)} dòng)" for row in result["majority_distribution"].head(3).itertuples(index=False))
    top_disagreement_labels = ", ".join(f"{row.label} ({int(row.disagreement_rows_with_label)} dòng)" for row in result["frequent_disagreement_labels"].head(3).itertuples(index=False))
    if not result["confusion_pairs"].empty:
        top_pair = result["confusion_pairs"].iloc[0]
        top_pair_text = f"{top_pair['label_a']} vs {top_pair['label_b']} ({int(top_pair['disagreement_rows'])} dòng)"
    else:
        top_pair_text = "không có cặp bất đồng"
    most_average = result["annotator_alignment"].sort_values("average_similarity_rank").iloc[0]
    outlier = result["annotator_alignment"].sort_values("outlier_rank").iloc[0]

    insight_lines = [
        f"- Fleiss' kappa của vòng sau cập nhật guideline là {fmt_num(fleiss['kappa'])}; Cohen's kappa trung bình từng cặp là {fmt_num(cohen_mean)}.",
        f"- Tỷ lệ dòng có bất đồng là {fmt_pct(disagreement_rate)}: {counts.get('partial_disagreement', 0)} dòng khác 1 phần và {counts.get('complete_disagreement', 0)} dòng khác hoàn toàn; có {result['rows_incomplete']} dòng thiếu nhãn.",
        f"- Các nhãn majority phổ biến nhất là {top_majority}.",
        f"- Các nhãn phát sinh bất đồng nhiều nhất là {top_disagreement_labels}; cặp bất đồng nhiều nhất là {top_pair_text}.",
        f"- Annotator giống trung bình nhóm nhất là {most_average['annotator']} (Cohen trung bình {fmt_num(most_average['avg_pairwise_cohen'])}, khớp majority của 2 người còn lại {fmt_pct(most_average['match_other2_consensus_rate'])}); annotator khác biệt nhất là {outlier['annotator']} (Cohen trung bình {fmt_num(outlier['avg_pairwise_cohen'])}, lệch khỏi majority của 2 người còn lại {fmt_pct(outlier['minority_when_other2_agree_rate'])}).",
    ]

    parts = [
        "# IAA Analysis After Guideline Update",
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "## Mục tiêu",
        "Phân tích mức nhất quán gán nhãn topic của 3 annotator (Nhung, Yen, Han) cho vòng pilot sau khi cập nhật guideline. Report này chỉ trình bày metric và phân tích phân phối/bất đồng, không bao gồm phần gợi ý cập nhật guideline.",
        "## Dữ liệu",
        f"Nguồn: `{AFTER_GUIDELINE_PATH.relative_to(REPO_ROOT)}`. File có {result['rows_total']} dòng, trong đó {result['rows_valid_3_annotators']} dòng đủ cả 3 nhãn annotator và {result['rows_incomplete']} dòng thiếu ít nhất 1 nhãn annotator.",
        "## Phương pháp",
        "- Fleiss' kappa được tính trên các dòng đủ cả 3 nhãn annotator.\n- Cohen's kappa được tính từng cặp annotator trên các dòng mà cả hai người trong cặp đều có nhãn.\n- `full_agreement` nghĩa là 3 người gán cùng một nhãn; `partial_disagreement` nghĩa là 2 người cùng nhãn và 1 người khác; `complete_disagreement` nghĩa là 3 người gán 3 nhãn khác nhau; `incomplete` nghĩa là thiếu ít nhất 1 nhãn annotator.\n- Universe nhãn gồm 18 topic labels của guideline để xác định nhãn chưa được dùng trong tập này.",
        "## Universe nhãn",
        md_table(label_rows, ["label_id", "label"]),
        "## Tổng quan chỉ số",
        md_table(overview, list(overview.columns)),
        "## Insight chính",
        "\n".join(insight_lines),
        result_to_markdown(result),
    ]
    return "\n\n".join(parts) + "\n"

# Load and analyze
pilot_after_df = read_pilot(AFTER_GUIDELINE_PATH)
LABEL_UNIVERSE, LABEL_TO_ID = build_label_universe(pilot_after_df)
result_after = analyze_dataset(
    name="Pilot after guideline update - Set 100 [2]",
    df=pilot_after_df,
    label_universe=LABEL_UNIVERSE,
    label_to_id=LABEL_TO_ID,
    row_scope="Toàn bộ file sau cập nhật guideline (100 dòng)",
)

print(f"Repo root: {REPO_ROOT}")
print(f"Input file: {AFTER_GUIDELINE_PATH.name} -> {pilot_after_df.shape[0]} rows, {pilot_after_df.shape[1]} columns")
print(f"Report path: {REPORT_PATH}")
print(f"Universe nhãn: {len(LABEL_UNIVERSE)} labels")

In [ ]:
# In kết quả phân tích chính trong notebook
print("=" * 96)
print(result_after["name"])
print("=" * 96)
print(f"Tổng dòng: {result_after['rows_total']} | Dòng đủ 3 nhãn: {result_after['rows_valid_3_annotators']} | Dòng thiếu nhãn: {result_after['rows_incomplete']}")
print(f"Nhãn được dùng: {result_after['labels_used_count']} | Nhãn không được dùng: {result_after['labels_unused_count']}")
if result_after["unused_labels"]:
    print("Nhãn không được dùng:", ", ".join(result_after["unused_labels"]))

fleiss_rows, cohen = format_metric_tables(result_after)
print("\nFleiss' kappa")
print(md_table(fleiss_rows, list(fleiss_rows[0].keys())))
print("\nCohen's kappa từng cặp")
print(md_table(cohen, list(cohen.columns)))
print("\nAnnotator giống trung bình nhóm / khác biệt so với nhóm")
alignment = format_annotator_alignment(result_after)
print(md_table(alignment, list(alignment.columns)))

In [ ]:
# In phân phối và bất đồng trong notebook
agreement = result_after["agreement_counts"].copy()
agreement["pct_all_rows"] = agreement["pct_all_rows"].map(fmt_pct)
print("Tần suất đồng thuận/bất đồng")
print(md_table(agreement, list(agreement.columns)))

majority_dist = result_after["majority_distribution"].copy()
majority_dist["pct_valid_rows"] = majority_dist["pct_valid_rows"].map(fmt_pct)
if result_after["rows_incomplete"]:
    majority_dist = pd.concat([
        majority_dist,
        pd.DataFrame([{"label": "INCOMPLETE_MISSING_LABELS", "rows": result_after["rows_incomplete"], "pct_valid_rows": "NA"}]),
    ], ignore_index=True)
print("\nPhân phối nhãn cấp dòng theo majority vote")
print(md_table(majority_dist, list(majority_dist.columns)))

assignment_dist = result_after["assignment_distribution"].copy()
assignment_dist["assignment_pct"] = assignment_dist["assignment_pct"].map(fmt_pct)
print("\nPhân phối nhãn theo toàn bộ lượt gán")
print(md_table(assignment_dist, list(assignment_dist.columns)))

frequent = result_after["frequent_disagreement_labels"][[
    "label_id", "label", "rows_with_label", "disagreement_rows_with_label", "disagreement_rate_when_label_appears", "disagreement_assignment_count"
]].copy()
frequent["disagreement_rate_when_label_appears"] = frequent["disagreement_rate_when_label_appears"].map(fmt_pct)
print("\nNhãn hay phát sinh bất đồng")
print(md_table(frequent, list(frequent.columns)))

pairs = result_after["confusion_pairs"].copy()
if not pairs.empty:
    pairs["pct_of_disagreement_rows"] = pairs["pct_of_disagreement_rows"].map(fmt_pct)
print("\nCặp nhãn hay bị bất đồng với nhau")
print(md_table(pairs, list(pairs.columns) if not pairs.empty else ["label_a", "label_b", "disagreement_rows", "pct_of_disagreement_rows"]))

In [ ]:
# Xuất report markdown mới, không ghi đè report cũ
REPORT_MD = build_report(result_after)
REPORT_PATH.write_text(REPORT_MD, encoding="utf-8")
print(f"Đã xuất report: {REPORT_PATH}")
print(f"Số ký tự report: {len(REPORT_MD):,}")